# Entrainement MKAN — Version Google Colab GPU

Copie de `MKAN/train.ipynb` optimisee pour GPU CUDA sur Google Colab (extension VSCode).  
**Kernel independant : n'interrompt pas l'entrainement local.**

**Ordre d'execution** :
1. *Colab Setup* — une seule fois par session
2. *Configuration Colab* — adapter `GITHUB_REPO`, `DRIVE_DATA`, `BATCH_SIZE`
3. Toutes les cellules suivantes dans l'ordre

> **Reprise automatique** : si la session Colab est coupee, re-executer depuis  
> la cellule *Imports* (§). Le checkpoint sur Drive sera rechargé automatiquement.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  COLAB SETUP — executer EN PREMIER (une fois par session)
# ════════════════════════════════════════════════════════════════════════
import os, subprocess, sys

# 1) Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2) Cloner le repo MKAN (la racine du repo = le package MKAN)
#    Le repo est clone dans /content/MKAN/ — ne pas changer ce chemin
GITHUB_REPO = 'https://github.com/Dr-MKO-4/MKAN'

if not os.path.exists('/content/MKAN'):
    subprocess.run(['git', 'clone', GITHUB_REPO, '/content/MKAN'], check=True)
else:
    print('Code deja present dans /content/MKAN')

# 3) Repertoire de travail = racine du package (contient __init__.py, train_colab.ipynb…)
#    /content/ est le parent → import MKAN fonctionne via ROOT ci-dessous
os.chdir('/content/MKAN')

# 4) Installer les dependances (PyTorch + CUDA fournis par Colab)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy', 'pandas', 'plotly', 'tqdm', 'pyarrow', 'scikit-learn'],
               check=True)

print('Setup termine — cwd :', os.getcwd())

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  CONFIGURATION COLAB — adapter avant de lancer
# ════════════════════════════════════════════════════════════════════════
import os

# ── Donnees : parquets sur Google Drive ──────────────────────────────────
# Uploader ces 3 fichiers dans ce dossier Drive avant de lancer :
#   featuresLog.parquet    (train  ~5,5 M tx)
#   val_features.parquet   (val    ~1,65 M tx)
#   test_features.parquet  (test   ~1,65 M tx)
DRIVE_DATA = '/content/drive/MyDrive/MKAN_data'   # <- adapter si necessaire

# ── Checkpoints : ecrits sur Drive (persistants entre sessions) ──────────
COLAB_CHECKPOINT_DIR = '/content/drive/MyDrive/MKAN_checkpoints'
os.makedirs(COLAB_CHECKPOINT_DIR, exist_ok=True)

# ── Device ────────────────────────────────────────────────────────────────
os.environ['MKAN_DEVICE'] = 'cuda'   # 'cpu' pour forcer le CPU

# ── DataLoader GPU ────────────────────────────────────────────────────────
NUM_WORKERS = 4
PIN_MEMORY  = True

# ── Batch size — T4 = 16 Go VRAM, on peut monter a 1024 ou 2048 ──────────
BATCH_SIZE = 1024   # remplacer 256 (local) par 1024 pour mieux saturer la T4

# ── Mixed Precision (AMP) — ~1.5-2x plus rapide sur GPU Volta/Ampere ─────
USE_AMP = True

print(f'DRIVE_DATA           : {DRIVE_DATA}')
print(f'COLAB_CHECKPOINT_DIR : {COLAB_CHECKPOINT_DIR}')
print(f'BATCH_SIZE={BATCH_SIZE}  NUM_WORKERS={NUM_WORKERS}  USE_AMP={USE_AMP}')

# Entraînement MKAN — Pipeline complet

**Flux** :
1. Chargement de trois simulations indépendantes (seeds distincts) générées par `generate_sim_dataset.py` via `MOMTSIM` :
   - **Train** — `MOMTSIM/config/featuresLog.parquet` (500 K clients, seed 1000, ~5,5 M tx)
   - **Val**   — `data/val_features.parquet`  (150 K clients, seed 1001, ~1,65 M tx)
   - **Test**  — `data/test_features.parquet` (150 K clients, seed 1002, ~1,65 M tx)
2. Prétraitement uniforme : remplissage NaN → transforms log (shifts ajustés sur le train) → normalisation z-score (μ, σ ajustés sur le train)
3. Construction des fenêtres glissantes W × 12 features par compte `nameOrig` (eq. 4.16)
4. Entraînement `MKANScorer` via `mkan_total_loss` (section 4.3.4)
5. Évaluation MCC / AUC-ROC (baseline XGBoost = 0,82)
6. Détection de dérive JS + extension de grille (section 4.4)
7. Élagage + régression symbolique → rapport d'audit COBAC (section 4.4.7)

In [ ]:
import sys, os

# cwd = /content/MKAN/  →  ROOT = /content/  →  import MKAN trouve /content/MKAN/
ROOT        = os.path.abspath('..')
MOMTSIM_DIR = os.path.join(ROOT, 'MOMTSIM')
for p in [ROOT, MOMTSIM_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from MKAN import (
    MKANScorer,
    js_divergence, detect_drift_region,
    extract_full_model_report,
)
from MKAN.MKAN_cuda import mkan_total_loss   # sous-package GPU (sans .item() DirectML)

# Detection GPU
_force = os.environ.get('MKAN_DEVICE', '')
if _force:
    DEVICE = torch.device(_force)
    print(f'Device force (MKAN_DEVICE) : {DEVICE}')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'Device : CUDA -- {torch.cuda.get_device_name(0)}')
else:
    DEVICE = torch.device('cpu')
    print('Device : CPU')

if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    print(f'cuDNN benchmark actif | VRAM : {torch.cuda.get_device_properties(0).total_memory // 1024**2} Mo')

# Valeurs par defaut si colab-config n'a pas ete executee
_g = globals()
if 'NUM_WORKERS' not in _g: NUM_WORKERS = 0
if 'PIN_MEMORY'  not in _g: PIN_MEMORY  = False
if 'USE_AMP'     not in _g: USE_AMP     = DEVICE.type == 'cuda'
if 'BATCH_SIZE'  not in _g: BATCH_SIZE  = 256

print('PyTorch', torch.__version__, '| CUDA', torch.cuda.is_available())
print('sys.path[0:2] :', sys.path[:2])

## 1. Configuration

In [ ]:
# ── Chemins (donnees sur Drive) ────────────────────────────────────────────
FEATURES_FILE  = os.path.join(DRIVE_DATA, 'featuresLog.parquet')
VAL_FILE       = os.path.join(DRIVE_DATA, 'val_features.parquet')
TEST_FILE      = os.path.join(DRIVE_DATA, 'test_features.parquet')
CHECKPOINT_DIR = COLAB_CHECKPOINT_DIR
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Features (eq. 3.8-3.19) ─────────────────────────────────────────────────
FEATURE_COLS = [
    'delta_B_orig',
    'delta_B_dest',
    'r1',
    'r2',
    'flag_anomalie',
    'delta_commission',
    'var_agent_split',
    'rho_rupture',
    'rho_refund',
    'v1h',
    'flag_nuit',
    'rho_nouveau',
]
TARGET_COL  = 'isFraud'
ACCOUNT_COL = 'nameOrig'
TIME_COL    = 'step'

# ── Architecture MKAN (section 4.2) ─────────────────────────────────────────
INPUT_SIZE  = len(FEATURE_COLS)   # 12
HIDDEN_SIZE = 32   # valeur par defaut ; peut etre optimisee par la recherche
W           = 10
M_GAUSS     = 8
K_FOURIER   = 2

# ── Hyperparamètres d'entrainement ───────────────────────────────────────────
# BATCH_SIZE est defini dans colab-config (1024) ou dans les imports (256)
LR  = 1e-3
N_EPOCHS = 40
LAM = 1e-2
MU1 = 1.0
MU2 = 0.5

JS_THRESHOLD = 0.05

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'BATCH_SIZE     : {BATCH_SIZE}')
print(f'FEATURES_FILE  : {FEATURES_FILE}')
print(f'CHECKPOINT_DIR : {CHECKPOINT_DIR}')

## 2. Chargement et exploration des données

In [ ]:
df = pd.read_parquet(FEATURES_FILE)

# Remplissage des NaN avant toute normalisation
# delta_commission et var_agent_split sont NaN pour les tx où la feature ne s'applique pas :
# → 0 est la valeur neutre sémantiquement correcte (aucune activité de mule / aucun split)
nan_counts = df[FEATURE_COLS].isna().sum()
if nan_counts.any():
    print("NaN détectés (remplacement par 0) :")
    print(nan_counts[nan_counts > 0])
    df[FEATURE_COLS] = df[FEATURE_COLS].fillna(0.0)

print(f"\nTransactions totales : {len(df):,}")
print(f"Taux de fraude global : {df[TARGET_COL].mean():.3f}")
print(f"Steps : {df[TIME_COL].min()} → {df[TIME_COL].max()}")
print(f"Comptes uniques (nameOrig) : {df[ACCOUNT_COL].nunique():,}")
if df['fraudScenario'].notna().any():
    print("\nRépartition par scénario :")
    print(df.loc[df[TARGET_COL], 'fraudScenario'].value_counts())

In [ ]:
# Statistiques des 12 features
df[FEATURE_COLS + [TARGET_COL]].describe().round(4)

## 2.5 Détection et transformation log — même protocole que MOMTSIM (§4.1, éq. 4.5)

`TopologyValidator` (MOMTSIM/src/viz.py) calcule $D_{KS}$ comme la distance maximale entre
la ECDF normalisée et $\Phi$ (éq. 4.5). Seuil : $D_{KS} \geq 0.15$ → transformation.

Transformation appliquée (identique à `apply_recommended_transforms()`) :
$$x' = \log\!\left(1 + \max(x - x_{\min},\, 0)\right)$$
i.e. décalage vers 0 si la feature est négative, puis $\log(1+\cdot)$.

In [ ]:
import sys as _sys, os as _os
_momtsim_src = _os.path.join(_os.path.abspath(".."), "MOMTSIM", "src")
if _momtsim_src not in _sys.path:
    _sys.path.insert(0, _momtsim_src)

from viz import TopologyValidator, BINARY_FEATURES

# ── Validation topologique sur le DataFrame complet (avant split) ──────────────
_validator = TopologyValidator(df, features=FEATURE_COLS)
_validator.normalize()           # éq. 4.1 — z-score interne au validateur
ks_results = _validator.ks_per_feature()   # éq. 4.5 — ECDF vs Φ, sample=5000

print(f"Test KS vs loi normale (éq. 4.5)  —  seuil D_KS ≥ 0.15\n")
print(f"  {'feature':25s}  {'D_KS':>6}  décision")
print(f"  {'-'*56}")
for col in FEATURE_COLS:
    if col in BINARY_FEATURES:
        print(f"  {col:25s}  {'—':>6}  ignoré (binaire)")
        continue
    ks = ks_results.get(col)
    flag = "✓ TRANSFORM" if ks is not None and ks >= 0.15 else "— OK"
    ks_str = f"{ks:.3f}" if ks is not None else "?"
    print(f"  {col:25s}  {ks_str:>6}  {flag}")

LOG_COLS = _validator.report.get("features_needing_transform", [])
print(f"\n→ {len(LOG_COLS)} features à transformer : {LOG_COLS}")

# LOG_SHIFTS : min de chaque feature sur le train, pour appliquer le même décalage à val/test
LOG_SHIFTS: dict[str, float] = {}
for col in LOG_COLS:
    x = df[col].values
    col_min = float(x.min())
    LOG_SHIFTS[col] = col_min
    shifted = x - col_min if col_min < 0 else x
    df[col] = np.log1p(shifted)

if LOG_COLS:
    print("\nStatistiques post-transformation :")
    print(df[LOG_COLS].describe().round(4).to_string())

## 3. Découpage train / val / test + Normalisation (section 4.1.1, eq. 4.1)

Trois simulations indépendantes (seeds distincts) pour éliminer tout biais de déplétion ATO :

| Split | Fichier source | Clients | Steps | Seed | ~Transactions |
|-------|---------------|---------|-------|------|---------------|
| **Train** | `MOMTSIM/config/featuresLog.parquet` | 500 000 | 1 440 | 1000 | 5,5 M |
| **Val**   | `data/val_features.parquet`          | 150 000 | 1 440 | 1001 | ~1,65 M |
| **Test**  | `data/test_features.parquet`         | 150 000 | 1 440 | 1002 | ~1,65 M |

Les fichiers val/test sont générés par `generate_sim_dataset.py` (à exécuter une seule fois).
La normalisation $\tilde{x}_{ij} = (x_{ij} - \mu_j) / (\sigma_j + \varepsilon)$ est ajustée **uniquement sur le train**.

In [ ]:
def _apply_log_transforms(d: pd.DataFrame) -> None:
    """Applique les mêmes transforms log que sur le train (shifts fixés sur LOG_SHIFTS)."""
    d[FEATURE_COLS] = d[FEATURE_COLS].fillna(0.0)
    for col in LOG_COLS:
        x = d[col].values
        train_min = LOG_SHIFTS[col]
        shifted = np.maximum(x - train_min, 0) if train_min < 0 else np.maximum(x, 0)
        d[col] = np.log1p(shifted)


for path, name in [(VAL_FILE, "val"), (TEST_FILE, "test")]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Fichier {name} introuvable : {os.path.abspath(path)}\n"
            "→ Générer les données avec :\n"
            "    python generate_sim_dataset.py sim_config_val.json\n"
            "    python generate_sim_dataset.py sim_config_test.json"
        )

df_train = df.copy()   # df a déjà les log-transforms appliquées (cellule précédente)

df_val  = pd.read_parquet(VAL_FILE)
df_test = pd.read_parquet(TEST_FILE)

_apply_log_transforms(df_val)
_apply_log_transforms(df_test)

print(f"Train : {len(df_train):,} tx  ({df_train[TARGET_COL].mean():.3f} fraude)"
      f"  — {df_train[ACCOUNT_COL].nunique():,} comptes")
print(f"Val   : {len(df_val):,} tx  ({df_val[TARGET_COL].mean():.3f} fraude)"
      f"  — {df_val[ACCOUNT_COL].nunique():,} comptes")
print(f"Test  : {len(df_test):,} tx  ({df_test[TARGET_COL].mean():.3f} fraude)"
      f"  — {df_test[ACCOUNT_COL].nunique():,} comptes")

# Normalisation fit sur le train uniquement (évite la fuite de données)
mu_train  = df_train[FEATURE_COLS].mean()
std_train = df_train[FEATURE_COLS].std().clip(lower=1e-6)

for d in [df_train, df_val, df_test]:
    d[FEATURE_COLS] = (d[FEATURE_COLS] - mu_train) / std_train

print("\nNormalisation appliquée.")

## 4. Construction des fenêtres glissantes (eq. 4.16)

Pour chaque compte `nameOrig`, on crée toutes les fenêtres de $W$ transactions consécutives.
Le label d'une fenêtre est l'étiquette `isFraud` de la **dernière** transaction de la fenêtre.

In [ ]:
from tqdm.auto import tqdm as _tqdm

def make_windows(df_split: pd.DataFrame, W: int, desc: str = "Fenêtres", leave: bool = True):
    """
    Fenêtres glissantes par compte (eq. 4.16) : (batch, W, 12) → label de la dernière tx.
    Comptes avec < W transactions sont ignorés (fenêtre impossible).
    leave=False supprime la barre après complétion (utile depuis la recherche heuristique).
    """
    X_list, y_list = [], []
    feat = df_split[FEATURE_COLS].values.astype(np.float32)
    targ = df_split[TARGET_COL].values.astype(np.float32)
    acct = df_split[ACCOUNT_COL].values
    step = df_split[TIME_COL].values

    _, unique_starts = np.unique(acct, return_index=True)
    unique_ends = np.append(unique_starts[1:], len(acct))

    for start, end in _tqdm(zip(unique_starts, unique_ends),
                             total=len(unique_starts),
                             desc=desc, unit="compte", leave=leave):
        n = end - start
        if n < W:
            continue
        order = np.argsort(step[start:end])
        Xacc = feat[start:end][order]
        yacc = targ[start:end][order]
        for i in range(n - W + 1):
            X_list.append(Xacc[i:i+W])
            y_list.append(yacc[i+W-1])

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    return X, y


for d in [df_train, df_val, df_test]:
    d.sort_values([ACCOUNT_COL, TIME_COL], inplace=True)
    d.reset_index(drop=True, inplace=True)

X_train, y_train = make_windows(df_train, W, desc="Train")
X_val,   y_val   = make_windows(df_val,   W, desc="Val  ")
X_test,  y_test  = make_windows(df_test,  W, desc="Test ")

print(f"\nFenêtres train : {len(X_train):,}  (fraude : {y_train.mean():.3f})")
print(f"Fenêtres val   : {len(X_val):,}  (fraude : {y_val.mean():.3f})")
print(f"Fenêtres test  : {len(X_test):,}  (fraude : {y_test.mean():.3f})")

In [ ]:
def to_loader(X, y, batch_size, shuffle):
    X_t = torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32))
    y_t = torch.from_numpy(np.ascontiguousarray(y, dtype=np.float32))
    ds  = TensorDataset(X_t, y_t)
    return DataLoader(
        ds,
        batch_size         = batch_size,
        shuffle            = shuffle,
        num_workers        = NUM_WORKERS,
        pin_memory         = PIN_MEMORY,
        persistent_workers = NUM_WORKERS > 0,
        prefetch_factor    = 2 if NUM_WORKERS > 0 else None,
    )

train_loader = to_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
val_loader   = to_loader(X_val,   y_val,   BATCH_SIZE, shuffle=False)
test_loader  = to_loader(X_test,  y_test,  BATCH_SIZE, shuffle=False)

## 5. Optimisation des hyperparamètres (recherche heuristique)

Algorithme génétique amélioré (Hao & Solnon, 2024 ; Yang Lu & Felix Zhan, 2024) :

- **Mutation adaptative** : taux de mutation décroissant (refroidissement, analogie recuit simulé)
- **Contrôle de la diversité** : injection d'individus aléatoires si la population converge prématurément
- **Composante EDA** : 1/3 des enfants générés selon un modèle probabiliste sur les meilleurs individus

**Hyperparamètres explorés** : `hidden_size`, `M`, `K`, `lam`, `mu1`, `mu2`, `lr`

> Chaque évaluation entraîne un `MKANScorer` pendant `N_EPOCHS_SEARCH` epochs et retourne le MCC sur `val_loader`. Les meilleurs hyperparamètres (`best_hp`) sont utilisés pour l'entraînement final (section 13).

In [ ]:
import sys as _sys

# Rechargement complet MKAN + MKAN.MKAN_cuda pour prendre en compte
# toute modification des .py depuis le debut de la session
for _k in list(_sys.modules.keys()):
    if _k.startswith("MKAN"):
        del _sys.modules[_k]

from MKAN import (
    MKANScorer,
    js_divergence, detect_drift_region,
    extract_full_model_report,
)
from MKAN.MKAN_cuda import (
    mkan_total_loss,
    build_mkan_fitness, RechercheHeuristique,
    ESPACE_MKAN_DEFAULT, ESPACE_MKAN_ETENDU,
)

print("MKAN + MKAN.MKAN_cuda recharges.")
print(f"  ESPACE_MKAN_DEFAULT  ({len(ESPACE_MKAN_DEFAULT)} HP) : {list(ESPACE_MKAN_DEFAULT.keys())}")
print(f"  ESPACE_MKAN_ETENDU   ({len(ESPACE_MKAN_ETENDU)} HP) : {list(ESPACE_MKAN_ETENDU.keys())}")

In [ ]:
# ── Choix de l'espace de recherche ────────────────────────────────────────────
# ESPACE_MKAN_DEFAULT  →  7 HP (hidden_size, M, K, lam, mu1, mu2, lr)
# ESPACE_MKAN_ETENDU   →  9 HP (+ W et batch_size)
ESPACE_CHOISI = ESPACE_MKAN_DEFAULT   # ← changer ici pour activer W et batch_size

# ── Parametres de la recherche ────────────────────────────────────────────────
N_EPOCHS_SEARCH   = 10     # epochs par individu — augmenter pour plus de precision
MAX_TRAIN_SAMPLES = 30_000
MAX_VAL_SAMPLES   = 10_000

# ── Fonction fitness GPU-optimisee (MKAN_cuda) ────────────────────────────────
# num_workers / pin_memory / use_amp : definis dans colab-config
fitness_fn = build_mkan_fitness(
    df_train           = df_train,
    df_val             = df_val,
    make_windows       = make_windows,
    device             = DEVICE,
    input_size         = INPUT_SIZE,
    n_epochs           = N_EPOCHS_SEARCH,
    default_W          = W,
    default_batch_size = BATCH_SIZE,
    max_train_samples  = MAX_TRAIN_SAMPLES,
    max_val_samples    = MAX_VAL_SAMPLES,
    num_workers        = NUM_WORKERS,
    pin_memory         = PIN_MEMORY,
    use_amp            = USE_AMP,
)

# ── Configuration de la recherche ─────────────────────────────────────────────
search = RechercheHeuristique(
    espace            = ESPACE_CHOISI,
    fitness_fn        = fitness_fn,
    n_generations     = 15,
    taille_population = 12,
    elite_size        = 2,
    tournament_size   = 3,
    mutation_rate     = 0.35,
    refroidissement   = 0.97,
    min_diversite     = 0.40,
    patience          = 5,
    maximize          = True,
)

budget = search.n_generations * search.taille_pop
print(f"Espace : {list(ESPACE_CHOISI.keys())}")
print(f"Budget : {budget} evaluations max  ({N_EPOCHS_SEARCH} epochs chacune)")
print(f"Sous-echantillonnage : train={MAX_TRAIN_SAMPLES:,}  val={MAX_VAL_SAMPLES:,}\n")

best_hp = search.fit()

print(f"\n{'='*55}")
print("Meilleurs hyperparametres trouves :")
for k, v in best_hp['params'].items():
    print(f"  {k:12s} = {v}")
print(f"MCC val = {best_hp['score']:.4f}")

In [ ]:
# Résumé tabulaire de la progression
cols_resume  = ['generation', 'score', 'diversite', 'mutation_rate',
                'hidden_size', 'M', 'K', 'lam', 'lr']
cols_present = [c for c in cols_resume if c in search.resume().columns]
print("Progression par génération :")
print(search.resume()[cols_present].to_string(index=False))

# Visualisations Plotly de la recherche génétique
search.plot_convergence().show()
search.plot_parameter_space().show()
search.plot_diversity().show()

## 5. Instanciation du modèle MKAN (section 4.2.4, eq. 4.16)

In [ ]:
bp = best_hp['params']

# Taille reelle du hidden — peut differer de la constante HIDDEN_SIZE
MODEL_HIDDEN_SIZE = int(bp.get('hidden_size', HIDDEN_SIZE))

model = MKANScorer(
    input_size  = INPUT_SIZE,
    hidden_size = MODEL_HIDDEN_SIZE,
    M           = int(bp.get('M', M_GAUSS)),
    K           = int(bp.get('K', K_FOURIER)),
    domain      = 1.0,
).to(DEVICE)

LR  = float(bp.get('lr',  LR))
LAM = float(bp.get('lam', LAM))
MU1 = float(bp.get('mu1', MU1))
MU2 = float(bp.get('mu2', MU2))

# Priorite de chargement des poids :
#   1) Checkpoint Drive existant (reprise apres coupure Colab)
#   2) Warm-start depuis la recherche heuristique
_ckpt_path = os.path.join(CHECKPOINT_DIR, 'best_mkan.pt')
if os.path.exists(_ckpt_path):
    model.load_state_dict(
        torch.load(_ckpt_path, map_location=DEVICE, weights_only=True))
    print(f'Reprise depuis checkpoint Drive : {_ckpt_path}')
else:
    _best = fitness_fn.get_meilleur()   # FIX: .get_meilleur() et non .meilleur
    if _best['state_dict'] is not None:
        model.load_state_dict(_best['state_dict'])
        print(f'Warm-start depuis la recherche (MCC={_best["score"]:.4f})')
    else:
        print('Initialisation aleatoire')

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'MODEL_HIDDEN_SIZE : {MODEL_HIDDEN_SIZE}')
print(f'Parametres entrainables : {n_params:,}')

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

## 6. Boucle d'entraînement (ℓ_total, section 4.3.4)

$$\ell_{total} = \ell_{pred} + \lambda\left(\mu_1 \sum_l |\Phi_l|_1 + \mu_2 \sum_l S(\Phi_l)\right)$$

In [ ]:
@torch.inference_mode()
def evaluate(model, loader):
    """
    Metriques : MCC (eq. 3.1), AUC-ROC (eq. 3.2), PR-AUC (eq. 3.6), Brier (eq. 3.7).
    """
    model.eval()
    all_scores, all_labels = [], []
    for X_batch, y_batch in loader:
        # FIX: device_type dynamique — pas de crash si DEVICE=cpu
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            out = model(X_batch.to(DEVICE, non_blocking=True))
        all_scores.append(out.float().cpu().numpy())
        all_labels.append(y_batch.numpy())

    scores = np.concatenate(all_scores)
    labels = np.concatenate(all_labels).astype(int)
    preds  = (scores >= 0.5).astype(int)

    tp = int(((preds == 1) & (labels == 1)).sum())
    tn = int(((preds == 0) & (labels == 0)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())

    num = tp * tn - fp * fn
    den = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    mcc = float(num / (den + 1e-12))

    n_pos = labels.sum()
    n_neg = len(labels) - n_pos
    if n_pos > 0 and n_neg > 0:
        sidx       = np.argsort(-scores)
        labs_s     = labels[sidx].astype(float)
        tpr        = np.concatenate([[0.0], np.cumsum(labs_s)   / n_pos])
        fpr        = np.concatenate([[0.0], np.cumsum(1-labs_s) / n_neg])
        auc        = float(abs(np.trapz(tpr, fpr)))
        cum_pos    = np.cumsum(labs_s)
        cum_n      = np.arange(1, len(labs_s)+1, dtype=float)
        prec_curve = np.concatenate([[1.0], cum_pos / cum_n])
        rec_curve  = np.concatenate([[0.0], cum_pos / n_pos])
        pr_auc     = float(abs(np.trapz(prec_curve, rec_curve)))
    else:
        auc = pr_auc = float('nan')

    prec  = float(tp / (tp + fp + 1e-12))
    rec   = float(tp / (tp + fn + 1e-12))
    f1    = float(2 * prec * rec / (prec + rec + 1e-12))
    brier = float(np.mean((scores - labels.astype(float)) ** 2))

    return dict(mcc=mcc, auc=auc, pr_auc=pr_auc, brier=brier,
                precision=prec, recall=rec, f1=f1)

In [ ]:
from tqdm.auto import tqdm as _tqdm

history = {'epoch': [], 'loss': [], 'pred_loss': [], 'reg': [],
           'l1': [], 'entropy': [],
           'val_mcc': [], 'val_auc': [], 'val_prauc': [], 'val_brier': []}

best_val_mcc = -1.0
best_epoch   = 0

# FIX: GradScaler avec device_type dynamique
# AMP desactive si DEVICE=cpu (GradScaler non supporte sur CPU)
_amp_active = USE_AMP and DEVICE.type == 'cuda'
scaler = torch.amp.GradScaler(device=DEVICE.type, enabled=_amp_active)

pbar = _tqdm(range(1, N_EPOCHS + 1), desc='Entrainement', unit='epoch')
for epoch in pbar:
    model.train()
    epoch_loss = epoch_pred = epoch_l1 = epoch_ent = 0.0
    n_batches  = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        # FIX: device_type dynamique
        with torch.amp.autocast(device_type=DEVICE.type, enabled=_amp_active):
            loss, pred_loss, reg_l1, reg_entropy = mkan_total_loss(
                model, X_batch, y_batch, lam=LAM, mu1=MU1, mu2=MU2)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        epoch_pred += pred_loss.item()
        epoch_l1   += reg_l1.item()
        epoch_ent  += reg_entropy.item()
        n_batches  += 1

    avg_loss = epoch_loss / n_batches
    avg_pred = epoch_pred / n_batches
    avg_l1   = epoch_l1  / n_batches
    avg_ent  = epoch_ent / n_batches

    history['epoch'].append(epoch)
    history['loss'].append(avg_loss)
    history['pred_loss'].append(avg_pred)
    history['reg'].append(avg_loss - avg_pred)
    history['l1'].append(avg_l1)
    history['entropy'].append(avg_ent)

    val_metrics = evaluate(model, val_loader)
    history['val_mcc'].append(val_metrics['mcc'])
    history['val_auc'].append(val_metrics['auc'])
    history['val_prauc'].append(val_metrics['pr_auc'])
    history['val_brier'].append(val_metrics['brier'])

    if val_metrics['mcc'] > best_val_mcc:
        best_val_mcc = val_metrics['mcc']
        best_epoch   = epoch
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'best_mkan.pt'))

    pbar.set_postfix({
        'loss':    f"{avg_loss:.4f}",
        'val_MCC': f"{val_metrics['mcc']:.4f}",
        'val_AUC': f"{val_metrics['auc']:.4f}",
    })

print(f'\nMeilleur val MCC = {best_val_mcc:.4f} (epoch {best_epoch})')

## 7. Visualisation de la convergence

In [ ]:
import importlib, sys as _sys
for _k in [_k for _k in _sys.modules if _k.startswith('MKAN')]:
    del _sys.modules[_k]
from MKAN import MKANVisualizer

# FIX: MODEL_HIDDEN_SIZE (taille réelle après optimisation HP)
viz = MKANVisualizer(
    model         = model,
    history       = history,
    feature_names = FEATURE_COLS,
    hidden_size   = MODEL_HIDDEN_SIZE,
)
print(f'MKANVisualizer pret — {len(viz.concat_labels)} entrees')

fig_dashboard = viz.plot_training_dashboard()
fig_dashboard.write_html(os.path.join(CHECKPOINT_DIR, 'dashboard.html'))
fig_dashboard.show()
print(f'Dashboard : {CHECKPOINT_DIR}/dashboard.html')

## 8. Évaluation finale sur le test

Charge le meilleur checkpoint (val MCC maximal).

In [ ]:
# Rechargement du meilleur checkpoint
model.load_state_dict(
    torch.load(os.path.join(CHECKPOINT_DIR, "best_mkan.pt"),
               map_location=DEVICE, weights_only=True))

test_metrics = evaluate(model, test_loader)
print("=== Résultats TEST ===")
for k, v in test_metrics.items():
    print(f"  {k:12s} : {v:.4f}")

print(f"\nBaseline XGBoost (Azamuke et al. 2025) : MCC = 0.82, AUC = 0.97")
delta_mcc = test_metrics['mcc'] - 0.82
print(f"Écart MKAN / XGBoost   : ΔMCC = {delta_mcc:+.4f}")

In [ ]:
# Collecte des predictions sur le jeu de test (coherent avec evaluate)
model.eval()
all_scores_t, all_labels_t = [], []
with torch.inference_mode():
    for X_batch, y_batch in test_loader:
        with torch.amp.autocast(device_type=DEVICE.type, enabled=_amp_active):
            out = model(X_batch.to(DEVICE, non_blocking=True))
        all_scores_t.append(out.float().cpu().numpy())
        all_labels_t.append(y_batch.numpy())

scores_t = np.concatenate(all_scores_t)
labels_t = np.concatenate(all_labels_t).astype(int)
preds_t  = (scores_t >= 0.5).astype(int)

tn = int(((preds_t == 0) & (labels_t == 0)).sum())
fp = int(((preds_t == 1) & (labels_t == 0)).sum())
fn = int(((preds_t == 0) & (labels_t == 1)).sum())
tp = int(((preds_t == 1) & (labels_t == 1)).sum())

cm_values   = [[tn, fp], [fn, tp]]
axis_labels = ['Legitime', 'Fraude']
annot_text  = [[str(v) for v in row] for row in cm_values]

fig_cm = go.Figure(go.Heatmap(
    z=cm_values, x=axis_labels, y=axis_labels,
    colorscale='Blues', text=annot_text, texttemplate='%{text}', showscale=True,
))
fig_cm.update_layout(
    title='Matrice de confusion — MKAN (test)',
    xaxis_title='Predit', yaxis_title='Reel', width=420, height=370,
)
fig_cm.write_html(os.path.join(CHECKPOINT_DIR, 'confusion_matrix.html'))
fig_cm.show()
print(f'VN={tn}  FP={fp}  FN={fn}  VP={tp}')

## 9. Détection de dérive JS (section 4.4, eq. 4.19)

Compare la distribution des scores sur le **val** (référence) avec celle sur le **test** (déploiement simulé).

$$JS(P \| Q) = \frac{1}{2}\left[KL(P \| M) + KL(Q \| M)\right], \quad M = \frac{P+Q}{2}$$

In [ ]:
# Pool de reference (val) et pool de deploiement (test) — derniere tx de chaque fenetre
X_val_t  = torch.from_numpy(np.ascontiguousarray(X_val,  dtype=np.float32)).to(DEVICE)
X_test_t = torch.from_numpy(np.ascontiguousarray(X_test, dtype=np.float32)).to(DEVICE)

pool_ref = X_val_t[:, -1, :]    # (N_val,  12)
pool_new = X_test_t[:, -1, :]   # (N_test, 12)

# Divergence JS par feature (eq. 4.19)
# js_divergence attend des histogrammes de MEME longueur — on discretise d'abord
# les echantillons bruts avec des bins communs sur [-domain, domain]
DRIFT_BINS   = 20
DRIFT_DOMAIN = 3.0    # couvre [-3sigma, +3sigma] apres normalisation
bin_edges = np.linspace(-DRIFT_DOMAIN, DRIFT_DOMAIN, DRIFT_BINS + 1)

js_scores = []
for j, fname in enumerate(FEATURE_COLS):
    ref_j = pool_ref[:, j].cpu().numpy()
    new_j = pool_new[:, j].cpu().numpy()
    hist_ref, _ = np.histogram(ref_j, bins=bin_edges)
    hist_new, _ = np.histogram(new_j, bins=bin_edges)
    js = js_divergence(hist_ref.astype(float), hist_new.astype(float))
    js_scores.append(js)
    flag = 'DERIVE' if js > JS_THRESHOLD else 'OK'
    print(f'  {fname:25s}  JS={js:.4f}  {flag}')

drift_features = [FEATURE_COLS[j] for j, js in enumerate(js_scores) if js > JS_THRESHOLD]
print(f'\nFeatures en derive (JS > {JS_THRESHOLD}) : {drift_features}')


In [ ]:
# Extension de grille (eq. 4.17–4.18) sur les features en dérive
# detect_drift_region retourne (region, js_val) où region = (xl, xr) ou None
if drift_features:
    print('Extension de grille sur les arêtes des portes MKAN...')
    gates_list    = [model.cell.forget_gate, model.cell.input_gate,
                     model.cell.candidate_gate, model.cell.output_gate]
    N_NEW_CENTERS = 4   # centres gaussiens insérés dans la région de dérive

    extended = False
    for fname in drift_features:
        feat_idx = FEATURE_COLS.index(fname)
        region, js_val = detect_drift_region(
            pool_ref[:, feat_idx].cpu().numpy(),
            pool_new[:, feat_idx].cpu().numpy(),
        )
        print(f'  {fname}: JS={js_val:.4f}, région={region}')

        if region is not None:
            for gate in gates_list:
                new_M = gate.extend_grid(region, N_NEW_CENTERS)
            extended = True
            print(f'    -> Grille étendue à M={new_M} centres')

    if extended:
        # Reconstruction de l'optimiseur après extension (nn.Parameter remplacé)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR * 0.1)
        print(f'\nOptimiseur reconstruit (lr={LR * 0.1:.1e})')
else:
    print('Aucune dérive détectée — extension de grille non nécessaire.')

## 10. Rapport d'audit COBAC — Élagage + Régression symbolique (section 4.4.7)

In [ ]:
from tqdm.auto import tqdm as _tqdm

model.eval()
N_AUDIT  = min(500, len(X_train))
audit_X  = torch.from_numpy(np.ascontiguousarray(X_train[:N_AUDIT], dtype=np.float32)).to(DEVICE)
concat_list = []

# FIX: MODEL_HIDDEN_SIZE (pas HIDDEN_SIZE qui peut différer après optimisation)
with torch.inference_mode():
    for b_start in _tqdm(range(0, N_AUDIT, 64), desc="Pool d'audit", unit='batch'):
        xb    = audit_X[b_start:b_start + 64]
        batch = xb.shape[0]
        h_t   = torch.zeros(batch, MODEL_HIDDEN_SIZE, device=DEVICE)
        c_t   = torch.zeros(batch, MODEL_HIDDEN_SIZE, device=DEVICE)
        for t in range(W - 1):
            h_t, c_t = model.cell(xb[:, t, :], h_t, c_t)
        concat_list.append(torch.cat([h_t, xb[:, W - 1, :]], dim=1).cpu())

x_pool_audit = torch.cat(concat_list, dim=0).to(DEVICE)
print(f'Pool audit : {x_pool_audit.shape}  '
      f'(attendu : [{N_AUDIT}, {MODEL_HIDDEN_SIZE + INPUT_SIZE}])')

In [ ]:
# Élagage + régression symbolique sur les 4 portes T-KAN (section 4.4.7)
# x_pool_audit : (N, hidden_size + input_size) — entrées réelles des portes
report = extract_full_model_report(
    model,
    x_pool        = x_pool_audit,
    feature_names = FEATURE_COLS,
    theta         = 1e-2,
    r2_threshold  = 0.99,
)

# Statistiques globales
n_active   = sum(len(edges) for edges in report.values())
n_symbolic = sum(1 for edges in report.values()
                 for e in edges if e['symbolifiable'])

print('=== Rapport d audit COBAC ===')
print(f'Arêtes actives   (|φ|₁ > θ=0.01)  : {n_active}')
print(f'Arêtes symbolifiables (R² ≥ 0.99) : {n_symbolic} / {n_active}')
print()

for gate_name, edges in report.items():
    if not edges:
        print(f'Porte {gate_name.upper():10s} — aucune arête active')
        continue
    print(f'Porte {gate_name.upper():10s} ({len(edges)} arêtes actives) :')
    for edge in edges[:5]:   # top 5 par importance L1
        symb = 'OK ' if edge['symbolifiable'] else '~  '
        print(f"  [{symb}] {edge['input']:15s} -> {edge['output']:12s}  "
              f"L1={edge['l1_importance']:.4f}  "
              f"R2={edge['r2']:.4f}  "
              f"{edge['formula']}")
    print()

In [ ]:
import json

torch.save({
    'model_state': model.state_dict(),
    'config': {
        'input_size':   INPUT_SIZE,
        'hidden_size':  MODEL_HIDDEN_SIZE,   # FIX: valeur reelle apres optimisation HP
        'W':            W,
        'M':            M_GAUSS,
        'K':            K_FOURIER,
        'feature_cols': FEATURE_COLS,
        'mu_train':     mu_train.to_dict(),
        'std_train':    std_train.to_dict(),
    },
    'test_metrics':  test_metrics,
    'best_val_mcc':  best_val_mcc,
}, os.path.join(CHECKPOINT_DIR, 'mkan_final.pt'))

with open(os.path.join(CHECKPOINT_DIR, 'audit_report.json'), 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)

print('Sauvegarde :')
print(f'  {CHECKPOINT_DIR}/mkan_final.pt         — modele + config + metriques')
print(f'  {CHECKPOINT_DIR}/audit_report.json      — rapport d audit COBAC')
print(f'  {CHECKPOINT_DIR}/training_curves.html   — courbes Plotly interactives')
print(f'  {CHECKPOINT_DIR}/confusion_matrix.html  — matrice de confusion Plotly')

## 11. Visualisations structurelles des aretes T-KAN (MKANVisualizer)

`viz` a ete instancie a la section 7. On utilise ici `x_pool_audit` (section 10) pour les methodes necessitant les entrees reelles des portes.

- `plot_edge_heatmap` — heatmap L1 par porte (eq. 2.19)
- `plot_edge_functions` — courbes phi_ij(x) avec decomposition Gaussienne + Fourier (section 4.2.2)
- `plot_pruning_summary` — bilan de sparsification par porte (eq. 4.28)

In [ ]:
# Courbes de perte decomposees et metriques individuelles
for title, method in [
    ('loss_curves',     viz.plot_loss_curves),
    ('regularization',  viz.plot_regularization_detail),
    ('val_metrics',     viz.plot_metrics),
]:
    fig = method()
    fig.write_html(os.path.join(CHECKPOINT_DIR, f'{title}.html'))
    fig.show()


In [ ]:
from tqdm.auto import tqdm as _tqdm

for gate_name in _tqdm(['forget', 'input', 'candidate', 'output'],
                        desc="Heatmaps", unit="porte"):
    fig_hm = viz.plot_edge_heatmap(gate_name, x_pool_audit)
    fig_hm.write_html(os.path.join(CHECKPOINT_DIR, f'heatmap_{gate_name}.html'))
    fig_hm.show()
    _tqdm.write(f"  Heatmap {gate_name} sauvegardée.")

In [ ]:
from tqdm.auto import tqdm as _tqdm

for gate_name in _tqdm(['forget', 'input', 'candidate', 'output'],
                        desc="Edge functions", unit="porte"):
    try:
        fig_fn = viz.plot_edge_functions(gate_name, x_pool_audit, theta=1e-2, top_k=6)
        fig_fn.write_html(os.path.join(CHECKPOINT_DIR, f'edge_functions_{gate_name}.html'))
        fig_fn.show()
    except ValueError as e:
        _tqdm.write(f"  Porte {gate_name} ignorée : {e}")

In [ ]:
# Bilan de sparsification : arêtes totales vs survivantes par porte (eq. 4.28)
fig_prune = viz.plot_pruning_summary(x_pool_audit, theta=1e-2)
fig_prune.write_html(os.path.join(CHECKPOINT_DIR, 'pruning_summary.html'))
fig_prune.show()
print(f"Bilan élagage sauvegardé : {CHECKPOINT_DIR}/pruning_summary.html")
